### 1) Normalización de nombres y tipos de columnas

- Se corrigen errores comunes en los nombres (p. ej. `0ppointment Type` → `Appointment Type`).
- Se eliminan espacios en columnas de texto y se convierten columnas numéricas a tipos numéricos.

In [3]:
# 02 - Limpieza de datos y primer modelo con Árbol de Decisión
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas

repo_root = Path.cwd() 
RAW_PATH = repo_root / 'data' / 'raw' / 'database_non-shows.xlsx'
PROCESSED_DIR= repo_root / 'data' / 'processed'
if not os.path.exists(RAW_PATH):
    RAW_PATH = os.path.join('..', 'data', 'raw', 'database_non-shows.xlsx')

# Cargar
print('Cargando dataset desde:', RAW_PATH)
df = pd.read_excel(RAW_PATH)
print('Dataset cargado. Filas, columnas:', df.shape)

# Vista rápida
display(df.head())
print('\nColumnas:')
print(df.columns.tolist())

# Mostrar nulos y duplicados
print('\nValores nulos por columna:')
print(df.isnull().sum())
print('\nDuplicados totales:', df.duplicated().sum())

Cargando dataset desde: ..\data\raw\database_non-shows.xlsx
Dataset cargado. Filas, columnas: (18587, 13)
Dataset cargado. Filas, columnas: (18587, 13)
Dataset cargado. Filas, columnas: (18587, 13)


,Appointment Type,Age,Sex,Insurance Type,Number of Diseases,Recent Hospitalization,Number of Medications,Hour,Day,Month,Creation to Assignment Interval,Number of Previous Attendance,Number of Previous Non-Attendance
0,0,90,1,3,4,0,6,9,1,8,58,7,1
1,0,92,1,4,1,0,0,13,2,5,61,3,0
2,0,92,1,4,2,0,0,13,4,7,33,2,0
3,0,81,0,3,4,0,0,16,0,6,1,4,1
4,0,84,1,3,1,0,0,10,3,1,19,2,0



Columnas:
['Appointment Type', 'Age', 'Sex', 'Insurance Type', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications', 'Hour', 'Day', 'Month', 'Creation to Assignment Interval', 'Number of Previous Attendance', 'Number of Previous Non-Attendance']

Valores nulos por columna:
Appointment Type                     0
Age                                  0
Sex                                  0
Insurance Type                       0
Number of Diseases                   0
Recent Hospitalization               0
Number of Medications                0
Hour                                 0
Day                                  0
Month                                0
Creation to Assignment Interval      0
Number of Previous Attendance        0
Number of Previous Non-Attendance    0
dtype: int64

Duplicados totales: 6


In [4]:
# Normalizar nombres comunes y eliminar espacios en strings
# Map de correcciones conocidas (añadir si detectas otros errores)
rename_map = {
    'Appointment Type': 'Appointment Type',
    'Appointment_Type': 'Appointment Type',
    'AppointmentType': 'Appointment Type',
    'Number of diseases': 'Number of Diseases',
    'Number of diseases ': 'Number of Diseases',
}

# Aplicar renombrado sólo si existen las keys
existing_renames = {k:v for k,v in rename_map.items() if k in df.columns}
if existing_renames:
    df.rename(columns=existing_renames, inplace=True)
    print('Se aplicaron renombres:', existing_renames)

# Eliminar espacios en columnas string
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype(str).str.strip()

# Intentar convertir columnas numéricas conocidas a numeric
num_cols_candidates = ['Age', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications',
                       'Hour', 'Creation to Assignment Interval', 'Number of Previous Attendance', 'Number of Previous Non-Attendance']
for c in num_cols_candidates:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('\nTipos después de normalización:')
print(df.dtypes)

Se aplicaron renombres: {'Appointment Type': 'Appointment Type'}

Tipos después de normalización:
Appointment Type                     int64
Age                                  int64
Sex                                  int64
Insurance Type                       int64
Number of Diseases                   int64
Recent Hospitalization               int64
Number of Medications                int64
Hour                                 int64
Day                                  int64
Month                                int64
Creation to Assignment Interval      int64
Number of Previous Attendance        int64
Number of Previous Non-Attendance    int64
dtype: object


### 2) Eliminación de registros inválidos y duplicados

Se aplican reglas basadas en el diccionario del dataset para identificar registros con valores fuera de rango (ej.: Age <18 o >120, Hour fuera de 0-23, contadores negativos). Luego guardamos el resultado en `df_sin_errores_digitacion` y después removemos outliers extremos si procede.

In [5]:
# Reglas para identificar valores inválidos (según diccionario)
condiciones = {
    'Appointment Type': ~df['Appointment Type'].isin([0, 1]) if 'Appointment Type' in df.columns else pd.Series(False, index=df.index),
    'Age': (df['Age'] < 18) | (df['Age'] > 120) if 'Age' in df.columns else pd.Series(False, index=df.index),
    'Sex': ~df['Sex'].isin([0, 1, 2]) if 'Sex' in df.columns else pd.Series(False, index=df.index),
    'Insurance Type': ~df['Insurance Type'].isin(range(0, 9)) if 'Insurance Type' in df.columns else pd.Series(False, index=df.index),
    'Number of Diseases': (df['Number of Diseases'] < 0) if 'Number of Diseases' in df.columns else pd.Series(False, index=df.index),
    'Recent Hospitalization': (df['Recent Hospitalization'] < 0) if 'Recent Hospitalization' in df.columns else pd.Series(False, index=df.index),
    'Number of Medications': (df['Number of Medications'] < 0) if 'Number of Medications' in df.columns else pd.Series(False, index=df.index),
    'Hour': (df['Hour'] < 0) | (df['Hour'] > 23) if 'Hour' in df.columns else pd.Series(False, index=df.index),
    'Day': ~df['Day'].between(0, 6) if 'Day' in df.columns else pd.Series(False, index=df.index),
    'Month': ~df['Month'].between(1, 12) if 'Month' in df.columns else pd.Series(False, index=df.index),
    'Creation to Assignment Interval': (df['Creation to Assignment Interval'] < 0) if 'Creation to Assignment Interval' in df.columns else pd.Series(False, index=df.index),
    'Number of Previous Attendance': (df['Number of Previous Attendance'] < 0) if 'Number of Previous Attendance' in df.columns else pd.Series(False, index=df.index),
    'Number of Previous Non-Attendance': (df['Number of Previous Non-Attendance'] < 0) if 'Number of Previous Non-Attendance' in df.columns else pd.Series(False, index=df.index),
}

# Contar y combinar mascaras
count_invalid = 0
mask_invalid = pd.Series(False, index=df.index)
for col_name, cond in condiciones.items():
    if isinstance(cond, pd.Series):
        n = cond.sum()
        if n > 0:
            print(f"{n} valores inválidos en regla: {col_name}")
        count_invalid += int(n)
        mask_invalid = mask_invalid | cond

print('\nTotal de registros marcados como inválidos (con posible solapamiento):', count_invalid)

# Eliminar filas inválidas
df_sin_errores_digitacion = df[~mask_invalid].copy()
print('\nFilas antes:', len(df), 'Filas después de eliminar inválidos:', len(df_sin_errores_digitacion))

# Eliminar duplicados exactos
dup_count = df_sin_errores_digitacion.duplicated().sum()
if dup_count > 0:
    print('\nDuplicados encontrados:', dup_count)
    df_sin_errores_digitacion = df_sin_errores_digitacion.drop_duplicates()

print('\nFilas después de drop_duplicates:', len(df_sin_errores_digitacion))

51 valores inválidos en regla: Age
129 valores inválidos en regla: Day
36 valores inválidos en regla: Month
1 valores inválidos en regla: Creation to Assignment Interval

Total de registros marcados como inválidos (con posible solapamiento): 217

Filas antes: 18587 Filas después de eliminar inválidos: 18391

Duplicados encontrados: 6

Filas después de drop_duplicates: 18385


### 3) Tratamiento de outliers

- Eliminaremos registros con `Creation to Assignment Interval` > 365 (configurable).
- Opcionalmente se puede activar eliminación por IQR para variables continuas (disponible en código).

In [6]:
# Copiar df limpio base
df_limpio = df_sin_errores_digitacion.copy()

# Remover Creation to Assignment Interval excesivo
if 'Creation to Assignment Interval' in df_limpio.columns:
    n_before = len(df_limpio)
    df_limpio = df_limpio[df_limpio['Creation to Assignment Interval'] <= 365]
    print(f"Se eliminaron {n_before - len(df_limpio)} registros con Creation to Assignment Interval > 365")

# Función opcional para remover outliers por IQR en columnas numéricas
def remove_outliers_iqr(df, cols, k=1.5):
    df2 = df.copy()
    for c in cols:
        if c in df2.columns:
            Q1 = df2[c].quantile(0.25)
            Q3 = df2[c].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - k * IQR
            upper = Q3 + k * IQR
            df2 = df2[(df2[c].isna()) | ((df2[c] >= lower) & (df2[c] <= upper))]
    return df2

# Usar con precaución; por defecto está comentado
# num_cols_to_check = ['Age', 'Number of Diseases', 'Number of Medications', 'Creation to Assignment Interval']
# df_limpio = remove_outliers_iqr(df_limpio, num_cols_to_check, k=1.5)

print('\nFilas finales en df_limpio:', len(df_limpio))

# Guardar df_limpio
os.makedirs(PROCESSED_DIR, exist_ok=True)
processed_path = os.path.join(PROCESSED_DIR, 'df_limpio.csv')
df_limpio.to_csv(processed_path, index=False)
print('Dataset limpio guardado en:', processed_path)

Se eliminaron 1 registros con Creation to Assignment Interval > 365

Filas finales en df_limpio: 18384
Dataset limpio guardado en: c:\Users\alejo\OneDrive\Documentos\SEMESTRE VIII\Repositorio-PDG\predictive-no-show-fvl\notebooks\data\processed\df_limpio.csv
Dataset limpio guardado en: c:\Users\alejo\OneDrive\Documentos\SEMESTRE VIII\Repositorio-PDG\predictive-no-show-fvl\notebooks\data\processed\df_limpio.csv
Dataset limpio guardado en: c:\Users\alejo\OneDrive\Documentos\SEMESTRE VIII\Repositorio-PDG\predictive-no-show-fvl\notebooks\data\processed\df_limpio.csv


---

## 4) Preparación para Modelado y primer Árbol de Decisión (sin balanceo)

- Convertiremos variables categóricas relevantes (Sex, Insurance Type, Day, Month) a dummies.
- Seleccionaremos features básicos y entrenaremos un `DecisionTreeClassifier` como baseline.
- No aplicamos balanceo de clases en esta primera iteración (tal como solicitaste).

In [7]:
# Preparación de features y entrenamiento de Decision Tree mejorado
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Selección de features
# -------------------------------
features_num = [c for c in ['Age', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications',
                             'Hour', 'Creation to Assignment Interval', 'Number of Previous Attendance',
                             'Number of Previous Non-Attendance'] if c in df_limpio.columns]

features_cat = [c for c in ['Sex', 'Insurance Type', 'Day', 'Month'] if c in df_limpio.columns]

print('Features numéricas:', features_num)
print('Features categóricas:', features_cat)

# -------------------------------
# Construir X, y
# -------------------------------
X = df_limpio[features_num + features_cat].copy()
y = df_limpio['Appointment Type']

# One-hot encode
if features_cat:
    X = pd.get_dummies(X, columns=features_cat, drop_first=True)

# Rellenar NA
X = X.fillna(X.median())

# Escalar numéricas (opcional pero mejora el modelo)
scaler = StandardScaler()
X[features_num] = scaler.fit_transform(X[features_num])

# -------------------------------
# Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None
)

# -------------------------------
# Modelo base mejorado
# -------------------------------
clf = DecisionTreeClassifier(
    random_state=42,
    max_depth=10,
    min_samples_split=50,
    min_samples_leaf=20,
    criterion='entropy'
)

clf.fit(X_train, y_train)

# -------------------------------
# Predicción y evaluación
# -------------------------------
y_pred = clf.predict(X_test)

print('\nAccuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred, digits=3))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

# -------------------------------
# Importancia de features
# -------------------------------
if hasattr(clf, 'feature_importances_'):
    fi = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print('\nTop 10 features por importancia:')
    display(fi.head(10))

# -------------------------------
# GridSearchCV para encontrar mejor árbol sin balancear
# -------------------------------
param_grid = {
    'max_depth': [6, 8, 10, 12, None],
    'min_samples_split': [10, 20, 50, 100],
    'min_samples_leaf': [5, 10, 20, 40],
    'criterion': ['gini', 'entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nMejores parámetros encontrados:")
print(grid.best_params_)

best_clf = grid.best_estimator_
y_pred_best = best_clf.predict(X_test)

print('\nAccuracy (GridSearch):', accuracy_score(y_test, y_pred_best))
print('\nClassification Report (GridSearch):\n')
print(classification_report(y_test, y_pred_best, digits=3))
print('\nConfusion Matrix (GridSearch):')
print(confusion_matrix(y_test, y_pred_best))


# ================================================================
# 🔥 ENTRENAMIENTO REPETIDO (10 seeds) → Escoger el mejor modelo
# ================================================================

print("\n================ ENTRENAMIENTO REPETIDO ==================\n")

results = []
N_REPETICIONES = 10

for seed in range(N_REPETICIONES):
    clf_rep = DecisionTreeClassifier(
        random_state=seed,
        max_depth=best_clf.max_depth,
        min_samples_split=best_clf.min_samples_split,
        min_samples_leaf=best_clf.min_samples_leaf,
        criterion=best_clf.criterion
    )
    
    clf_rep.fit(X_train, y_train)
    y_pred_rep = clf_rep.predict(X_test)

    acc = accuracy_score(y_test, y_pred_rep)
    results.append((seed, acc, clf_rep))

    print(f"🏁 Iteración {seed+1}/{N_REPETICIONES} → Accuracy: {acc:.4f}")

# Elegir el mejor modelo repetido
best_seed, best_acc, best_model_rep = max(results, key=lambda x: x[1])

print("\n🎉 Mejor modelo encontrado tras repetir 10 veces:")
print("Seed:", best_seed)
print("Accuracy:", best_acc)

# Reporte final del mejor modelo repetido
y_pred_final = best_model_rep.predict(X_test)

print("\nClassification Report (MEJOR MODELO REPETIDO):\n")
print(classification_report(y_test, y_pred_final, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))


Features numéricas: ['Age', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications', 'Hour', 'Creation to Assignment Interval', 'Number of Previous Attendance', 'Number of Previous Non-Attendance']
Features categóricas: ['Sex', 'Insurance Type', 'Day', 'Month']

Accuracy: 0.7772640739733478

Classification Report:

              precision    recall  f1-score   support

           0      0.834     0.836     0.835      2477
           1      0.660     0.656     0.658      1200

    accuracy                          0.777      3677
   macro avg      0.747     0.746     0.746      3677
weighted avg      0.777     0.777     0.777      3677


Confusion Matrix:
[[2071  406]
 [ 413  787]]

Top 10 features por importancia:


Number of Previous Non-Attendance    0.299900
Hour                                 0.108156
Month_11                             0.098607
Month_12                             0.097558
Month_10                             0.095595
Number of Previous Attendance        0.072468
Day_6                                0.057211
Day_5                                0.044226
Number of Medications                0.034824
Number of Diseases                   0.028960
dtype: float64


Mejores parámetros encontrados:
{'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 20, 'min_samples_split': 100}

Accuracy (GridSearch): 0.7846070165896111

Classification Report (GridSearch):

              precision    recall  f1-score   support

           0      0.812     0.885     0.847      2477
           1      0.708     0.578     0.637      1200

    accuracy                          0.785      3677
   macro avg      0.760     0.731     0.742      3677
weighted avg      0.778     0.785     0.778      3677


Confusion Matrix (GridSearch):
[[2191  286]
 [ 506  694]]

================ ENTRENAMIENTO REPETIDO ==================

🏁 Iteración 1/10 → Accuracy: 0.7846
🏁 Iteración 2/10 → Accuracy: 0.7871
🏁 Iteración 3/10 → Accuracy: 0.7871
🏁 Iteración 4/10 → Accuracy: 0.7871
🏁 Iteración 2/10 → Accuracy: 0.7871
🏁 Iteración 3/10 → Accuracy: 0.7871
🏁 Iteración 4/10 → Accuracy: 0.7871
🏁 Iteración 2/10 → Accuracy: 0.7871
🏁 Iteración 3/10 → Accuracy: 0.7871
🏁 Iteración 4/10 → Accur

In [8]:
# ============================================================
# MODELOS: RANDOM FOREST + XGBOOST (SIN BALANCEAR)
# ============================================================

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# ============================================================
# PREPARACIÓN DE FEATURES
# ============================================================

features_num = [c for c in ['Age', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications',
                             'Hour', 'Creation to Assignment Interval', 'Number of Previous Attendance',
                             'Number of Previous Non-Attendance'] if c in df_limpio.columns]

features_cat = [c for c in ['Sex', 'Insurance Type', 'Day', 'Month'] if c in df_limpio.columns]

print("Features numéricas:", features_num)
print("Features categóricas:", features_cat)

# Construir X, y
X = df_limpio[features_num + features_cat].copy()
y = df_limpio["Appointment Type"]

# One-hot encoding
if features_cat:
    X = pd.get_dummies(X, columns=features_cat, drop_first=True)

# Rellenar NA
X = X.fillna(X.median())

# Escalar numéricas
scaler = StandardScaler()
X[features_num] = scaler.fit_transform(X[features_num])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# 1. RANDOM FOREST (SIN BALANCEO)
# ============================================================

print("\n===============================================")
print(" ENTRENANDO RANDOM FOREST (SIN BALANCEO)")
print("===============================================\n")

rf_param_grid = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [None, 10, 30, 50],
    "min_samples_split": [2, 10, 50],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True],
    "criterion": ["gini", "entropy"]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)


rf_grid.fit(X_train, y_train)

print("Mejores parámetros RF:", rf_grid.best_params_)

rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_test)

print("\nAccuracy (Random Forest):", accuracy_score(y_test, rf_pred))
print("\nClassification Report (Random Forest):\n")
print(classification_report(y_test, rf_pred, digits=3))

print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_test, rf_pred))

# Importancia de features
fi_rf = pd.Series(rf_best.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features RF:")
display(fi_rf.head(10))


Features numéricas: ['Age', 'Number of Diseases', 'Recent Hospitalization', 'Number of Medications', 'Hour', 'Creation to Assignment Interval', 'Number of Previous Attendance', 'Number of Previous Non-Attendance']
Features categóricas: ['Sex', 'Insurance Type', 'Day', 'Month']

 ENTRENANDO RANDOM FOREST (SIN BALANCEO)

Mejores parámetros RF: {'bootstrap': True, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 600}
Mejores parámetros RF: {'bootstrap': True, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 600}
Mejores parámetros RF: {'bootstrap': True, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 600}

Accuracy (Random Forest): 0.8058199619254828

Classification Report (Random Forest):

              precision    recall  f1-score   support

         

Number of Previous Non-Attendance    0.173157
Creation to Assignment Interval      0.088941
Hour                                 0.087090
Number of Previous Attendance        0.085928
Age                                  0.085827
Number of Diseases                   0.055017
Month_11                             0.054054
Month_12                             0.051418
Month_10                             0.048809
Day_6                                0.032380
dtype: float64

In [9]:
# ============================================================
# 2. XGBOOST (SIN BALANCEO)
# ============================================================

print("\n===============================================")
print(" ENTRENANDO XGBOOST (SIN BALANCEO)")
print("===============================================\n")

xgb_param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 5, 10],
    "reg_lambda": [1, 2, 5],
    "reg_alpha": [0, 0.1, 0.5]
}

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_grid = GridSearchCV(
    xgb,
    xgb_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)

print("Mejores parámetros XGB:", xgb_grid.best_params_)

xgb_best = xgb_grid.best_estimator_
xgb_pred = xgb_best.predict(X_test)

print("\nAccuracy (XGBoost):", accuracy_score(y_test, xgb_pred))
print("\nClassification Report (XGBoost):\n")
print(classification_report(y_test, xgb_pred, digits=3))

print("\nConfusion Matrix (XGBoost):")
print(confusion_matrix(y_test, xgb_pred))

# Importancia de features XGB
fi_xgb = pd.Series(xgb_best.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features XGB:")
display(fi_xgb.head(10))



 ENTRENANDO XGBOOST (SIN BALANCEO)

Mejores parámetros XGB: {'colsample_bytree': 1.0, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'reg_alpha': 0.1, 'reg_lambda': 2, 'subsample': 1.0}

Accuracy (XGBoost): 0.8118031003535491

Classification Report (XGBoost):

              precision    recall  f1-score   support

           0      0.835     0.899     0.865      2477
           1      0.751     0.632     0.687      1200

    accuracy                          0.812      3677
   macro avg      0.793     0.766     0.776      3677
weighted avg      0.808     0.812     0.807      3677


Confusion Matrix (XGBoost):
[[2226  251]
 [ 441  759]]

Top 10 features XGB:
Mejores parámetros XGB: {'colsample_bytree': 1.0, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'reg_alpha': 0.1, 'reg_lambda': 2, 'subsample': 1.0}

Accuracy (XGBoost): 0.8118031003535491

Classification Report (XGBoost):

        

Month_10                             0.154289
Month_11                             0.144108
Month_12                             0.144094
Day_5                                0.060673
Day_6                                0.059157
Number of Previous Non-Attendance    0.055007
Insurance Type_1                     0.024414
Hour                                 0.022678
Sex_2                                0.021962
Day_2                                0.020826
dtype: float32